In [0]:
from pyspark.sql import functions as F

In [0]:
df_bmp = spark.table("PUC_Sprint_2.anp.silver_bmp")
df_bdep = spark.table("PUC_Sprint_2.anp.silver_bdep")
df_bar = spark.table("PUC_Sprint_2.anp.silver_bar")

pocos_bmp = df_bmp.select("poco").distinct()
pocos_bdep = df_bdep.select("poco").distinct()
total_pocos_bmp = pocos_bmp.count()
match_poco = pocos_bmp.join(pocos_bdep, on="poco", how="inner").count()
print(f"Poços do BMP com correspondência no BDEP: {match_poco}/{total_pocos_bmp} ({match_poco/total_pocos_bmp:.1%})")

campos_bmp = df_bmp.select("campo").distinct()
campos_bar = df_bar.select("campo").distinct()
total_campos_bmp = campos_bmp.count()
match_campo = campos_bmp.join(campos_bar, on="campo", how="inner").count()
print(f"Campos do BMP com correspondência no BAR: {match_campo}/{total_campos_bmp} ({match_campo/total_campos_bmp:.1%})")

In [0]:
campos_sem_match_bmp = campos_bmp.join(campos_bar, on="campo", how="left_anti")

# Para cada campo sem match, pega o ano mais recente de produção no BMP
producao_recente_sem_match = (df_bmp
    .join(campos_sem_match_bmp, on="campo", how="inner")
    .groupBy("campo")
    .agg(F.max("ano").alias("ultimo_ano_producao"))
)

producao_recente_sem_match.groupBy(
    F.when(F.col("ultimo_ano_producao") >= 2020, "produziu em 2020+")
     .otherwise("parou antes de 2020")
     .alias("categoria")
).count().show()

In [0]:
sem_match_mas_ativo = (producao_recente_sem_match
    .filter(F.col("ultimo_ano_producao") >= 2020)
    .select("campo"))

sem_match_mas_ativo.show(30, truncate=False)

In [0]:
nomes_bar = [r["campo"] for r in df_bar.select("campo").distinct().collect()]

def achar_parecidos(nome_bmp, lista_bar):
    return [nb for nb in lista_bar if nome_bmp in nb or nb in nome_bmp]

for nome in ["CIDADE DE SAO MIGUEL DOS CAMPOS", "SAO MIGUEL DOS CAMPOS", "SOCORRO EXTENSAO", "UIRAPURU SUDOESTE", "NORTE DE FAZENDA CARUACU"]:
    parecidos = achar_parecidos(nome, nomes_bar)
    print(f"{nome} -> {parecidos if parecidos else 'nenhum parecido encontrado'}")

In [0]:
campos_sem_match_lista = [r["campo"] for r in sem_match_mas_ativo.collect()]

resultados = []
for nome in campos_sem_match_lista:
    parecidos = achar_parecidos(nome, nomes_bar)
    resultados.append((nome, len(parecidos) > 0))

df_resultado_fuzzy = spark.createDataFrame(resultados, ["campo", "tem_parecido"])
df_resultado_fuzzy.groupBy("tem_parecido").count().show()

In [0]:
sem_nenhum_parecido = df_resultado_fuzzy.filter(~F.col("tem_parecido")).select("campo")

producao_desses = (df_bmp
    .join(sem_nenhum_parecido, on="campo", how="inner")
    .groupBy("campo")
    .agg(F.sum("producao_oleo_m3").alias("producao_total")))

producao_desses.orderBy(F.desc("producao_total")).show(20, truncate=False)

In [0]:
sem_nenhum_parecido = df_resultado_fuzzy.filter(~F.col("tem_parecido")).select("campo")

producao_desses = (df_bmp
    .join(sem_nenhum_parecido, on="campo", how="inner")
    .groupBy("campo")
    .agg(F.sum("producao_oleo_m3").alias("producao_total")))

producao_desses.orderBy(F.desc("producao_total")).show(20, truncate=False)

# Contexto: quanto isso representa do total nacional produzido no período
producao_total_geral = df_bmp.agg(F.sum("producao_oleo_m3")).collect()[0][0]
producao_total_sem_match = producao_desses.agg(F.sum("producao_total")).collect()[0][0]
print(f"\nProdução dos campos sem match no BAR: {producao_total_sem_match:,.0f} m³")
print(f"Produção total do BMP: {producao_total_geral:,.0f} m³")
print(f"Representa {producao_total_sem_match/producao_total_geral:.1%} do total")

In [0]:
producao_desses.orderBy(F.desc("producao_total")).show(20, truncate=False)

In [0]:
df_bar.filter(F.col("bacia").contains("SANTOS")).select("campo", "bacia").distinct().show(50, truncate=False)